In [ ]:
from glob import glob
import os
import pandas as pd
import h5py
import numpy as np
import openslide
from PIL import Image
import matplotlib.pyplot as plt
import anndata as ad
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        
organ=['Ovary', 'Breast', 'Pancreas', 'Lymphoid', 'Prostate', 'Kidney',
       'Lung', 'Skin', 'Bowel', 'Heart', 'Brain', 'Bone', 'Liver',
       'Cervix', 'Lymph node', 'Bladder', 'Eye', 'Uterus']
st_technology=['Visium HD', "Visium HD 3'", 'Xenium', 'Visium',
       'Spatial Transcriptomics']
for o in organ:
    for t in st_technology:
        create_dir(f'../../data/marker_gene_spatial_expression/label/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/image/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/image_5x/{t}/{o}')
        
mpp=20
image_size=512

In [ ]:
data_path='../../data/spatialTranscriptome/'
meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')

slide=openslide.open_slide(f'{data_path}wsis/{meta_df.loc[51]["image_filename"]}')
# h5ad 파일 읽기
adata = ad.read_h5ad(f'{data_path}st/{meta_df.loc[51]["id"]}.h5ad')

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')
from tqdm import tqdm

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# ===== Pass 1: 전체 슬라이드에서 gene별 global percentile 계산 =====
gene_values = {g: [] for g in marker_genes}
valid_slide_ids = []

for idx in tqdm(range(len(meta_df)), desc='Pass 1: Collecting gene stats'):
    row = meta_df.iloc[idx]
    sample_id = row['id']

    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        continue

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    missing = [g for g in marker_genes if g not in gene_names]
    if len(missing) > 0:
        continue

    valid_slide_ids.append(sample_id)

    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray()
    else:
        expr_matrix = np.array(adata.X[:, gene_indices])

    for gi, gene in enumerate(marker_genes):
        # 0이 아닌 발현값만 수집 (sparse 데이터 특성상 0이 대부분)
        vals = expr_matrix[:, gi]
        gene_values[gene].append(vals)

# gene별 p1, p99 계산
gene_p1 = {}
gene_p99 = {}
for gene in marker_genes:
    all_vals = np.concatenate(gene_values[gene])
    gene_p1[gene] = np.percentile(all_vals, 1)
    gene_p99[gene] = np.percentile(all_vals, 99)
    print(f'{gene:8s} | p1={gene_p1[gene]:.4f}, p99={gene_p99[gene]:.4f}')

print(f'\nValid slides: {len(valid_slide_ids)} / {len(meta_df)}')

# global stats 저장 (재사용 가능)
global_stats = np.array([[gene_p1[g], gene_p99[g]] for g in marker_genes])
np.save(f'{data_path}global_gene_stats_p1_p99.npy', global_stats)
print(f'Saved: {data_path}global_gene_stats_p1_p99.npy')

In [ ]:
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay
from concurrent.futures import ThreadPoolExecutor

# ===== Pass 2: Global stats 기반 정규화 + 패치 생성 =====

# global stats 로드 (Pass 1에서 저장한 파일 또는 메모리)
if 'global_stats' not in dir():
    global_stats = np.load(f'{data_path}global_gene_stats_p1_p99.npy')
    gene_p1 = {g: global_stats[i, 0] for i, g in enumerate(marker_genes)}
    gene_p99 = {g: global_stats[i, 1] for i, g in enumerate(marker_genes)}

target_mag = mpp  # 20 (target 20x)
context_mag = 5   # 5x context patch
patch_size = image_size  # 512
heatmap_size = 128
heatmap_ratio = patch_size // heatmap_size  # 4
min_spots = 1

save_image_dir = '../../data/marker_gene_spatial_expression/image'
save_image_5x_dir = '../../data/marker_gene_spatial_expression/image_5x'
save_label_dir = '../../data/marker_gene_spatial_expression/label'

# 이미 처리된 슬라이드 스킵 (재실행 시 이어서 처리)
import glob as glob_module
existing_files = set()
for f in glob_module.glob(f'{save_image_dir}/**/*.tiff', recursive=True):
    basename = os.path.splitext(os.path.basename(f))[0]
    parts = basename.rsplit('_', 2)
    if len(parts) == 3:
        existing_files.add(parts[0])

skipped = []
processed = []

for idx in tqdm(range(len(meta_df)), desc='Pass 2: Processing slides'):
    row = meta_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']
    patch_count = 0

    if sample_id in existing_files:
        continue

    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        skipped.append((sample_id, 'h5ad not found'))
        continue

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    missing = [g for g in marker_genes if g not in gene_names]
    if len(missing) > 0:
        skipped.append((sample_id, f'missing genes: {missing}'))
        continue

    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    native_mag = float(str(row['magnification']).replace('x', '').replace('X', ''))
    downsample_20x = native_mag / target_mag
    downsample_5x = native_mag / context_mag

    full_patch_20x = int(patch_size * downsample_20x)
    full_patch_5x = int(patch_size * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)

    hm_w = target_w // heatmap_ratio
    hm_h = target_h // heatmap_ratio

    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x

    hm_coords_x = coords_20x_x / heatmap_ratio
    hm_coords_y = coords_20x_y / heatmap_ratio

    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray()
    else:
        expr_matrix = np.array(adata.X[:, gene_indices])

    points = np.column_stack([hm_coords_x, hm_coords_y])
    tri = Delaunay(points)

    grid_y, grid_x = np.mgrid[0:hm_h, 0:hm_w]
    grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

    wsi_heatmaps = np.zeros((15, hm_h, hm_w), dtype=np.float32)

    for gi, gene in enumerate(marker_genes):
        interp = LinearNDInterpolator(tri, expr_matrix[:, gi], fill_value=0)
        heatmap = interp(grid_points).reshape(hm_h, hm_w)
        heatmap = np.clip(heatmap, 0, None)

        # global p1~p99 clip 후 min-max 정규화
        p1, p99 = gene_p1[gene], gene_p99[gene]
        heatmap = np.clip(heatmap, p1, p99)
        if p99 > p1:
            heatmap = (heatmap - p1) / (p99 - p1)
        else:
            heatmap = np.zeros_like(heatmap)
        wsi_heatmaps[gi] = heatmap.astype(np.float32)

    # 패치 분할 및 저장
    n_patches_x = target_w // patch_size
    n_patches_y = target_h // patch_size

    def save_patch(px, py):
        patch_x0 = px * patch_size
        patch_y0 = py * patch_size

        in_patch = (
            (coords_20x_x >= patch_x0) & (coords_20x_x < patch_x0 + patch_size) &
            (coords_20x_y >= patch_y0) & (coords_20x_y < patch_y0 + patch_size)
        )
        if in_patch.sum() < min_spots:
            return False

        hm_x = px * heatmap_size
        hm_y = py * heatmap_size
        if hm_x + heatmap_size > hm_w or hm_y + heatmap_size > hm_h:
            return False

        label_patch = wsi_heatmaps[:, hm_y:hm_y + heatmap_size, hm_x:hm_x + heatmap_size]

        # 20x 패치
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        # 5x 패치 (동일 중심)
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x))
        y5 = np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x))

        img_region_5x = slide.read_region((int(x5), int(y5)), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        np.save(f'{save_label_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_patch)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | {patch_count} patches saved')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')